Wikipedia page views (30 days before release) - Wikimedia API
Wikipedia page edits (30 days before release) - Wikimedia API

In [4]:
!pip install pandas

  Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached numpy-2.3.5-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl (10.7 MB)
Using cached numpy-2.3.5-cp313-cp313-macosx_14_0_arm64.whl (5.1 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [8]:
import requests
from datetime import datetime, timedelta
import pandas as pd

In [14]:
# The Wikimedia Pageviews API only has data from July 2015 onwards. 
movie_name = 'The Definite Maybe'
release_date = '1997-07-04'

In [15]:
# 30 days of pageviews for the movie
end_recent = datetime.now() - timedelta(days=1)
start_recent = end_recent - timedelta(days=30)

start_date_recent = start_recent.strftime('%Y%m%d')
end_date_recent = end_recent.strftime('%Y%m%d')

print(f"wikipedia page title: {movie_name}")
print(f"Release Date: {release_date} ")
print(f"Period: {start_date_recent} to {end_date_recent}")

wikipedia page title: The Definite Maybe
Release Date: 1997-07-04 
Period: 20251108 to 20251208


In [16]:
# Fetch recent pageviews
url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/{movie_name}/daily/{start_date_recent}/{end_date_recent}"

headers = {'User-Agent': 'MoviePageViewsAnalysis/1.0 (Educational purposes)'}
response = requests.get(url, headers=headers)

if response.status_code == 200: # API request successful
    data = response.json()
    
    # Extract pageview data
    pageviews = []
    for item in data['items']:
        pageviews.append({
            'date': datetime.strptime(item['timestamp'], '%Y%m%d%H').date(),
            'views': item['views']
        })
    
    # Create DataFrame
    df_views = pd.DataFrame(pageviews)

else:
    print(f"Error: {response.status_code}")
    print(response.text)

Error: 404
{"detail":"The date(s) you used are valid, but we either do not have data for those date(s), or the project you asked for is not loaded yet. Please check documentation for more information","method":"get","status":404,"title":"Not Found","type":"about:blank","uri":"/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/The%20Definite%20Maybe/daily/20251108/20251208"}


In [46]:
# Display full dataset
print(f"Total views: {df_views['views'].sum()}")
print(f"Average daily views: {df_views['views'].mean():.0f}")
print(f"Peak views: {df_views['views'].max()}")
print(df_views.head(100))

Total views: 319667
Average daily views: 10312
Peak views: 17565
          date  views
0   2025-10-20  13209
1   2025-10-21   9726
2   2025-10-22   8576
3   2025-10-23   8264
4   2025-10-24   8890
5   2025-10-25  10194
6   2025-10-26  11858
7   2025-10-27  10526
8   2025-10-28   9124
9   2025-10-29   8683
10  2025-10-30   8450
11  2025-10-31   9310
12  2025-11-01   9568
13  2025-11-02  10013
14  2025-11-03   9693
15  2025-11-04   9026
16  2025-11-05   9332
17  2025-11-06   9733
18  2025-11-07  10552
19  2025-11-08  11683
20  2025-11-09  11536
21  2025-11-10  10541
22  2025-11-11   9657
23  2025-11-12   9877
24  2025-11-13   8765
25  2025-11-14   9664
26  2025-11-15  10909
27  2025-11-16  17565
28  2025-11-17  13016
29  2025-11-18  11717
30  2025-11-19  10010


## Wikipedia Page Edits

In [48]:
# Get Wikipedia page title from movie name
import wikipedia

try:
    page = wikipedia.page(movie_name)
    article_title = page.title
    print(f"Wikipedia article: {article_title}")
except wikipedia.exceptions.DisambiguationError as e:
    print(f"Multiple pages found. Using first option: {e.options[0]}")
    article_title = e.options[0]
except wikipedia.exceptions.PageError:
    print(f"Page not found for '{movie_name}'")
    article_title = movie_name

Page not found for 'Jurassic World Rebirth'


In [49]:
# Fetch page edits using Wikipedia API
# The Wikimedia REST API doesn't have edit count by date, so we'll use MediaWiki API

# Build the API URL for revision history
api_url = "https://en.wikipedia.org/w/api.php"

# Convert date range to timestamps
start_timestamp = start_recent.strftime('%Y-%m-%dT00:00:00Z')
end_timestamp = end_recent.strftime('%Y-%m-%dT23:59:59Z')

params = {
    'action': 'query',
    'titles': article_title,
    'prop': 'revisions',
    'rvstart': end_timestamp,
    'rvend': start_timestamp,
    'rvlimit': 'max',
    'rvprop': 'timestamp|user|size',
    'format': 'json'
}

headers = {
    'User-Agent': 'MoviePageViewsAnalysis/1.0 (Educational purposes)'
}

print(f"Fetching edits from {start_recent.date()} to {end_recent.date()}...")
response_edits = requests.get(api_url, params=params, headers=headers)

if response_edits.status_code == 200:
    data_edits = response_edits.json()
    pages = data_edits.get('query', {}).get('pages', {})
    
    # Get the page data (there should be only one page)
    page_data = next(iter(pages.values()))
    
    if 'revisions' in page_data:
        revisions = page_data['revisions']
        
        # Process revisions
        edits_list = []
        for rev in revisions:
            timestamp = datetime.strptime(rev['timestamp'], '%Y-%m-%dT%H:%M:%SZ')
            edits_list.append({
                'date': timestamp.date(),
                'timestamp': timestamp,
                'user': rev.get('user', 'Unknown'),
                'size': rev.get('size', 0)
            })
        
        # Create DataFrame
        df_edits = pd.DataFrame(edits_list)
        
        print(f"\nTotal edits in period: {len(df_edits)}")
        print(f"Unique editors: {df_edits['user'].nunique()}")
        
        # Group by date
        edits_by_date = df_edits.groupby('date').size().reset_index(name='edit_count')
        print(f"\nEdits by date (first 10 days):")
        print(edits_by_date.head(10))
    else:
        print("No revisions found in this period")
        df_edits = pd.DataFrame()
else:
    print(f"Error: {response_edits.status_code}")
    print(response_edits.text)

Fetching edits from 2025-10-20 to 2025-11-19...

Total edits in period: 76
Unique editors: 20

Edits by date (first 10 days):
         date  edit_count
0  2025-10-20           7
1  2025-10-21           6
2  2025-10-22           1
3  2025-10-24          10
4  2025-10-25           2
5  2025-10-26          17
6  2025-10-27           3
7  2025-10-28           1
8  2025-10-29           1
9  2025-11-01           3

Total edits in period: 76
Unique editors: 20

Edits by date (first 10 days):
         date  edit_count
0  2025-10-20           7
1  2025-10-21           6
2  2025-10-22           1
3  2025-10-24          10
4  2025-10-25           2
5  2025-10-26          17
6  2025-10-27           3
7  2025-10-28           1
8  2025-10-29           1
9  2025-11-01           3


In [ ]:
# Display detailed edit information
if len(df_edits) > 0:
    print("\nEdit Statistics:")
    print(f"Average edits per day: {len(df_edits) / 30:.1f}")
    print(f"\nMost active editors:")
    print(df_edits['user'].value_counts().head(10))
    
    print("\nSample of recent edits:")
    display(df_edits[['date', 'timestamp', 'user', 'size']].head(15))
else:
    print("No edit data available")

In [5]:
df = pd.read_csv('/Users/kumar/Downloads/us_movies_with_reviews.csv')
df.head(20)

,tconst,primaryTitle,startYear,title,region,isOriginalTitle,release_date,reviews
0,tt0062336,The Tango of the Widower and Its Distorting Mi...,2020,The Tango of the Widower and Its Distorting Mi...,US,0,2020-02-21,[]
1,tt0069049,The Other Side of the Wind,2018,The Other Side of the Wind,US,0,2018-11-02,"[{'title': 'The last film by Orson Welles, fin..."
2,tt0070596,Socialist Realism,2023,Socialist Realism,US,0,2023-09-23,[]
3,tt0093119,Grizzly II: Revenge,2020,Grizzly II: The Predator,US,0,2020-02-17,"[{'title': 'The Long Lost NOW FOUND Sequel', '..."
4,tt0100275,The Wandering Soap Opera,2017,The Wandering Soap Opera,US,0,2018-09-06,[]
5,tt0116991,Mariette in Ecstasy,2019,Mariette in Ecstasy,US,0,1996-07-04,[]
6,tt0120589,The Surgeon of the Rusty Knife,2022,The Surgeon of the Rusty Knife,US,0,2022-09-01,"[{'title': 'A real history', 'body': ""Before t..."
7,tt0137204,Joe Finds Grace,2017,Joe Finds Grace,US,0,2017-09-21,[{'title': 'Clever ethereal heart touching lov...
8,tt0137818,Housesitter: The Night They Saved Siegfried's ...,2018,Housesitter: The Night They Saved Siegfried's ...,US,0,2020-10-02,"[{'title': 'Oh well', 'body': ''}, {'title': '..."
9,tt0164115,Nine Ball,2023,Nine Ball,US,0,2023-03-25,[]


In [10]:
# Install SPARQLWrapper for querying Wikidata
!pip install SPARQLWrapper


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import time
import ssl
import certifi

def get_cast_members(imdb_id):
    """
    Query Wikidata for cast members using IMDb ID
    Returns a list of actor names
    """
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.setReturnFormat(JSON)
    
    query = f"""
    SELECT ?actor ?actorLabel WHERE {{
      ?movie wdt:P345 "{imdb_id}".   # IMDb ID
      ?movie wdt:P161 ?actor.        # P161 = cast member
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
    }}
    """
    
    sparql.setQuery(query)
    
    try:
        # requests to avoid SSL issues
        import requests
        
        headers = {
            'User-Agent': 'MovieCastAnalysis/1.0 (Educational purposes)',
            'Accept': 'application/json'
        }
        
        response = requests.get(
            "https://query.wikidata.org/sparql",
            params={'query': query, 'format': 'json'},
            headers=headers
        )
        
        if response.status_code == 200:
            results = response.json()
            actors = [result['actorLabel']['value'] for result in results['results']['bindings']]
            return actors if actors else []
        else:
            print(f"Error {response.status_code} for {imdb_id}")
            return []
    except Exception as e:
        print(f"Error querying {imdb_id}: {e}")
        return []

Testing with tt0062336:
Cast members: []
Cast members: []


In [17]:
# Test with a few movies from the dataframe
print("Testing with movies from the dataframe:\n")
for i in range(min(5, len(df))):
    imdb_id = df['tconst'].iloc[i]
    title = df['primaryTitle'].iloc[i]
    cast = get_cast_members(imdb_id)
    print(f"{i+1}. {title} ({imdb_id}): {len(cast)} cast members")
    if cast:
        print(f"   Sample cast: {cast[:3]}")
    time.sleep(1)  # Rate limiting

Testing with movies from the dataframe:

1. The Tango of the Widower and Its Distorting Mirror (tt0062336): 0 cast members
1. The Tango of the Widower and Its Distorting Mirror (tt0062336): 0 cast members
2. The Other Side of the Wind (tt0069049): 38 cast members
   Sample cast: ['Stéphane Audran', 'John Huston', 'Claude Chabrol']
2. The Other Side of the Wind (tt0069049): 38 cast members
   Sample cast: ['Stéphane Audran', 'John Huston', 'Claude Chabrol']
3. Socialist Realism (tt0070596): 1 cast members
   Sample cast: ['Jaime Vadell']
3. Socialist Realism (tt0070596): 1 cast members
   Sample cast: ['Jaime Vadell']
4. Grizzly II: Revenge (tt0093119): 8 cast members
   Sample cast: ['John Rhys-Davies', 'George Clooney', 'Charlie Sheen']
4. Grizzly II: Revenge (tt0093119): 8 cast members
   Sample cast: ['John Rhys-Davies', 'George Clooney', 'Charlie Sheen']
5. The Wandering Soap Opera (tt0100275): 6 cast members
   Sample cast: ['Mauricio Pešutić', 'Patricia Rivadeneira', 'Francisco R

In [18]:
# Apply to all rows in the dataframe
# Note: This will take some time as we need to query Wikidata for each movie
# Wikidata has rate limits, so we add a small delay between requests

print(f"Processing {len(df)} movies...")
print("This may take a while. Progress updates every 10 movies.")

cast_members_list = []
for idx, imdb_id in enumerate(df['tconst']):
    cast = get_cast_members(imdb_id)
    cast_members_list.append(cast)
    
    # Progress update every 10 movies
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df)} movies...")
    
    # Rate limiting - wait 0.5 seconds between requests to be respectful
    time.sleep(0.5)

# Add as new column
df['cast_members'] = cast_members_list
print("\nDone! Cast members added to 'cast_members' column.")
print(f"\nSample results:")
print(df[['tconst', 'primaryTitle', 'cast_members']].head())

Processing 66496 movies...
This may take a while. Progress updates every 10 movies.


KeyboardInterrupt: 

In [2]:
import pandas as pd

In [4]:
df1 = pd.read_csv('/Users/kumar/Downloads/us_movies_with_reviews_1.csv')
df1['cast_members'].head(100)

0                                                    []
1     ['Stéphane Audran', 'John Huston', 'Claude Cha...
2                                      ['Jaime Vadell']
3     ['John Rhys-Davies', 'George Clooney', 'Charli...
4     ['Mauricio Pešutić', 'Patricia Rivadeneira', '...
                            ...                        
95                                                   []
96                                                   []
97                                                   []
98                                                   []
99                                                   []
Name: cast_members, Length: 100, dtype: object